# Прогноз загрузки трамвайных маршрутов

Стартовый ноутбук для Kaggle. Данные должны находиться в текущей рабочей директории.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('.')
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'

train = pd.read_csv(TRAIN_PATH, sep=';', encoding='utf-8')
test = pd.read_csv(TEST_PATH, sep=';', encoding='utf-8')
+
train.shape, test.shape

In [ ]:
def make_hourly_labels(frame):
    data = frame.copy()
    data['timestamp'] = pd.to_datetime(data['tran_date_time'], errors='coerce')
    data['route'] = data['ngpt_route'].str.extract(r'(\d+)')[0].astype('Int64')
    data['date'] = data['timestamp'].dt.date
    data['hour'] = data['timestamp'].dt.hour
    data = data[data['validation_result'].eq(1)]
    return (data.groupby(['route', 'date', 'hour'], as_index=False)
                .size()
                .rename(columns={'size': 'boardings'}))

train_labels = make_hourly_labels(train)
test_labels = make_hourly_labels(test)
train_labels.head()

In [ ]:
# Полная сетка для прогноза ноября и декабря 2025 года.
routes = sorted(train_labels['route'].dropna().unique())
dates = pd.date_range('2025-11-01', '2025-12-31', freq='D')
forecast_index = pd.MultiIndex.from_product(
    [routes, dates.date, range(24)], names=['route', 'date', 'hour']
).to_frame(index=False)
forecast_index.head(), forecast_index.shape